# Motion from Scratch

_How to go from pixels to motion._ Build a video engine from a single pixel up to a rebuilt motion-graphics intro, one layer at a time.

## How to use this notebook

1. **Save your own copy first:** File → Save a copy in Drive. Your code lives in this notebook, so the copy is how you keep your progress.
2. **Run the setup cell** below once per session. It downloads the course and checks that Node is available (Colab includes it).
3. **For each layer:** read the chapter, write your code in the cell that starts with `%%writefile`, run it, then run the test cell and the pictures cell.
4. **New session?** Run setup again, then re-run your `%%writefile` cells (Runtime → Run before) so your earlier layers are back on disk.

The course map, interactive guide and repo: [https://nalediym.github.io/motion-from-scratch/](https://nalediym.github.io/motion-from-scratch/) · [https://github.com/nalediym/motion-from-scratch](https://github.com/nalediym/motion-from-scratch)

**Stuck?** Ask a study buddy that gives hints, not answers: [open Claude](https://claude.ai/new?q=You're%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20(github.com%2Fnalediym%2Fmotion-from-scratch)%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Ask%20me%20questions%20first.%20If%20I'm%20still%20stuck%2C%20give%20one%20hint%20at%20a%20time%3A%20point%20at%20the%20test's%20expected%20vs%20actual%20values%2C%20then%20ask%20a%20concept%20question%2C%20then%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%20using%20different%20numbers%2C%20then%20point%20at%20the%20line%20to%20look%20at.%20Never%20go%20further%20than%20that.%20Explain%20what's%20happening%20under%20the%20hood%20(bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it).%20I'm%20just%20getting%20started.%20Start%20by%20asking%20what%20I'm%20working%20on.%20I'll%20paste%20my%20code%20and%20test%20output%20next.) · [open ChatGPT](https://chatgpt.com/?q=You're%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20(github.com%2Fnalediym%2Fmotion-from-scratch)%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Ask%20me%20questions%20first.%20If%20I'm%20still%20stuck%2C%20give%20one%20hint%20at%20a%20time%3A%20point%20at%20the%20test's%20expected%20vs%20actual%20values%2C%20then%20ask%20a%20concept%20question%2C%20then%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%20using%20different%20numbers%2C%20then%20point%20at%20the%20line%20to%20look%20at.%20Never%20go%20further%20than%20that.%20Explain%20what's%20happening%20under%20the%20hood%20(bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it).%20I'm%20just%20getting%20started.%20Start%20by%20asking%20what%20I'm%20working%20on.%20I'll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
#@title Setup: run once per session
import os, subprocess, glob
if not os.path.exists("motion-from-scratch") and not os.getcwd().endswith("motion-from-scratch"):
    subprocess.run(["git", "clone", "-q", "https://github.com/nalediym/motion-from-scratch"], check=True)
if not os.getcwd().endswith("motion-from-scratch"):
    os.chdir("motion-from-scratch")
node = subprocess.run(["node", "--version"], capture_output=True, text=True)
if node.returncode != 0:
    print("Installing Node...")
    subprocess.run("apt-get -qq update && apt-get -qq install -y nodejs > /dev/null", shell=True)
    node = subprocess.run(["node", "--version"], capture_output=True, text=True)
print("Course ready. Node", node.stdout.strip())

from IPython.display import Image, display
def pictures(layer):
    """Render a layer's test pictures with your code and show them."""
    out = subprocess.run(["node", "tools/pictures.js", layer], capture_output=True, text=True)
    print(out.stdout or out.stderr)
    for p in sorted(glob.glob(f"out/{layer}-*.png")):
        display(Image(p))

## Layer 00 · Pixels

_A framebuffer is memory you can see._

**You'll write:** `createFrame`, `pixelIndex`, `setPixel`, `getPixel`, `clearFrame` in `src/00-pixels.js`
**Check it:** `node tests/run.js 00`, or open `tests.html`
**Time:** one or two sittings

Every frame of the KineticIntro you'll build in layer 09 is 1080 × 1920 pixels. Before anything can move, you need somewhere to put those pixels and a way to change them one at a time. That place is called a **framebuffer**, and by the end of this layer you'll have written one.

### Under the hood

#### A pixel is three or four numbers

Your screen is a grid of tiny lights. Each one mixes red, green and blue light, and how bright each color glows is a number from 0 (off) to 255 (full). Why 255? Each channel gets one **byte**: 8 bits, which can hold 2⁸ = 256 different values, 0 through 255.

```text
  [255,   0,   0]  red          [  0,   0,   0]  black (all off)
  [255, 255, 255]  white        [ 91, 140, 255]  Motion Studio's blue
```

A fourth number, **alpha**, says how opaque the pixel is: 255 is solid, 0 is invisible. Alpha doesn't matter yet, but layer 01 is all about it. So one pixel is 4 bytes: **R, G, B, A**.

#### Memory is one long line

A picture is 2-D, but computer memory is a single long row of bytes. So you have to decide how to lay the grid out in a line. Nearly everything (browsers, image files, GPUs) uses the same answer: **row by row, left to right**, 4 bytes per pixel.

Here's a frame 4 pixels wide and 3 tall, with each cell showing where that pixel starts in the line:

```text
            x = 0   x = 1   x = 2   x = 3
          ┌───────┬───────┬───────┬───────┐
  y = 0   │   0   │   4   │   8   │  12   │
          ├───────┼───────┼───────┼───────┤
  y = 1   │  16   │  20   │  24   │  28   │
          ├───────┼───────┼───────┼───────┤
  y = 2   │  32   │  36   │  40   │  44   │
          └───────┴───────┴───────┴───────┘

  data: [ R G B A  R G B A  R G B A  R G B A | R G B A ... ]
          └ (0,0) ┘└ (1,0) ┘└ (2,0) ┘└ (3,0) ┘ └ (0,1) ...
          index 0   index 4   index 8   index 12  index 16
```

Look at the numbers for a while before writing `pixelIndex`. Going one step right adds the same amount each time. Going one step down adds a different, bigger amount. What is each amount, and why? If you can say that out loud, the formula writes itself.

#### Why a Uint8ClampedArray

JavaScript has **typed arrays**: fixed-size blocks of raw numbers, much faster than normal arrays because every slot is the same type. `Uint8ClampedArray` means:

- **Uint8**: each slot is an unsigned 8-bit integer, 0–255. Exactly one channel.
- **Clamped**: if you store something out of range, it's pinned to the nearest edge instead of wrapping around. Store 300 and you get 255; store −5 and you get 0; store 127.6 and it rounds to 128.

The browser uses this exact type for the pixels of a canvas (`ImageData.data`), which is why you're using it. Compare it with its unclamped cousin `Uint8Array`, where 300 wraps around to 44 (300 − 256). For colors, wrapping is a disaster: a pixel that should be "a bit brighter than white" would turn nearly black.

#### The edge of the world

What should happen if someone calls `setPixel(frame, 4, 0, ...)` on a frame that's 4 pixels wide? There is no column 4. But the index formula will still happily compute a number, and that number lands somewhere real:

```text
  setPixel(frame, x = 4, y = 0)  on a 4-wide frame

          x = 0   x = 1   x = 2   x = 3   (x = 4?)
  y = 0   ░░░░░░  ░░░░░░  ░░░░░░  ░░░░░░  ← you meant here
  y = 1   ██████  ░░░░░░  ░░░░░░  ░░░░░░
          └─ but the formula points here: (0, 1)
```

This "wrap-around" bug is real and common: shapes that hang off the right edge reappear on the left, one row down. Your `setPixel` must check bounds itself instead of trusting the formula. Later layers will draw things that slide off-screen all the time, so this check is load-bearing.

#### Getting it on screen

Your frame is just numbers in memory. To see it, `lib/present.js` (provided) copies them into a `<canvas>`:

```text
  your frame.data ──► new ImageData(data, w, h) ──► ctx.putImageData(...)
        (CPU memory)        (browser wrapper)          │
                                                       ▼
                     display ◄── GPU ◄── compositor ◄── canvas
               (refreshes ~60 times a second)
```

You don't need to write that part. The important idea is that the screen only ever shows what's in memory, and drawing just means changing numbers.

#### How big is a frame?

Your final video is 1080 × 1920, 4 bytes per pixel. Work it out before you read on: how many bytes is one frame? How many is 6 seconds at 30 frames per second?

<details><summary>Answer</summary>

One frame is 1080 × 1920 × 4 = 8,294,400 bytes, about 8.3 MB. 180 frames is about 1.5 GB. Yet the MP4 you export in layer 10 will be a few megabytes. Hold on to that gap; it's what video compression is for.

</details>

### Your turn

Work through `src/00-pixels.js` top to bottom. Each function's comment is its spec. Some questions to guide you, in order:

1. **createFrame.** How many bytes does a `width × height` frame need? What does a brand-new typed array contain? What counts as a "positive whole number", and is `2.5` one?
2. **pixelIndex.** Use the grid above. What do you multiply `y` by to skip whole rows? Why the final `× 4`?
3. **setPixel.** What order do you do things in: round, check bounds, compute index, write? What happens if you compute the index before checking bounds? What should alpha be when the color only has three numbers?
4. **getPixel.** Why return a *new* array instead of a piece of `frame.data`? (Hint: what happens if the caller changes it?)
5. **clearFrame.** You could call `setPixel` for every pixel. That works. Is there a way that touches `frame.data` directly with a single loop? Which is faster, and why?

Run `node tests/run.js 00` after each function. A failing test tells you exactly which pixel was wrong.

### Predict, then run

Write your guess down before you check. The point is to notice where your mental model differs from the machine.

1. `createFrame(1080, 1920).data.length` is what number?
2. You call `setPixel(f, 0, 0, [300, -5, 127.6])`. What does `getPixel(f, 0, 0)` return?
3. In the browser console, run `const a = new Uint8ClampedArray(4); a[1.5] = 200; a`. What's in `a`?

<details><summary>Answers</summary>

1. 8,294,400 (1080 × 1920 × 4).
2. `[255, 0, 128, 255]`. Clamped, rounded, and alpha filled in.
3. `[0, 0, 0, 0]`. A typed array only has whole-number slots, and a write to a fractional index like `1.5` is silently thrown away: no error, no change. This is why `setPixel` must round coordinates. Forget it, and drawing at x = 2.5 quietly does nothing.

Bonus: try `a[0] = 127.5; a[1] = 128.5`. Both become 128. Clamped arrays round exact halves to the nearest even number ("banker's rounding"), so rounding errors don't all lean the same way.

</details>

### Break it

Once your tests pass, break things on purpose and explain what you see. Undo each change afterwards.

1. **Swap x and y** in `pixelIndex`. Run the tests, then open `tests.html` and look at the XOR pattern. Which tests catch it, and what happened to the picture?
2. **Delete the bounds check** in `setPixel`. Which test fails? Draw a pixel at `(width, 0)` and find where it actually landed.
3. **Replace `Uint8ClampedArray` with `Uint8Array`** in `createFrame`. Which test fails, and what number did 300 become?

### Checkpoint

- `node tests/run.js 00` shows 11 of 11 passing.
- `tests.html` shows a checkerboard and a blue-and-orange XOR pattern under Layer 00.
- You can explain, without looking, why pixel (3, 2) in a 10-wide frame starts at index 92.

### In the real world

- The browser's own version of your frame is [`ImageData`](https://developer.mozilla.org/en-US/docs/Web/API/ImageData). Same layout, same `Uint8ClampedArray`.
- Video cards keep the image being shown in a region of memory literally called the framebuffer, scanned out to the display many times a second.
- Remotion doesn't touch pixels directly. It draws with the browser, then takes a screenshot of each frame, which is a full framebuffer like yours. You'll see that in layer 10.

### Study buddy prompts

Paste one of these into Claude Code, Codex or ChatGPT. The repo's `CLAUDE.md` and `AGENTS.md` already tell coding agents not to hand you answers.

- "I'm on layer 00 of Motion from Scratch. Don't give me code. Ask me questions until I can explain why the pixel index is (y × width + x) × 4."
- "Here's my setPixel. Don't fix it. Tell me which test case would break it, and let me find the bug."
- "Quiz me on Uint8ClampedArray versus Uint8Array with three predict-the-output questions."

### Going further (optional)

- Write `fillFrameFast(frame, color)` using `Uint32Array` over the same buffer (`new Uint32Array(frame.data.buffer)`) to set a whole pixel in one write. Time it against `clearFrame` on a 1080 × 1920 frame. Then look up **endianness** to understand why the byte order inside that 32-bit number looks backwards.


### ✏️ Your code for layer 00

Edit the cell below, then run it. The first line saves it as `src/00-pixels.js`. Then run the tests and pictures under it.

**Stuck?** Ask a study buddy that gives hints, not answers: [open Claude](https://claude.ai/new?q=You're%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20(github.com%2Fnalediym%2Fmotion-from-scratch)%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Ask%20me%20questions%20first.%20If%20I'm%20still%20stuck%2C%20give%20one%20hint%20at%20a%20time%3A%20point%20at%20the%20test's%20expected%20vs%20actual%20values%2C%20then%20ask%20a%20concept%20question%2C%20then%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%20using%20different%20numbers%2C%20then%20point%20at%20the%20line%20to%20look%20at.%20Never%20go%20further%20than%20that.%20Explain%20what's%20happening%20under%20the%20hood%20(bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it).%20I'm%20on%20layer%2000%20(Pixels)%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Start%20by%20asking%20what%20I'm%20working%20on.%20I'll%20paste%20my%20code%20and%20test%20output%20next.) · [open ChatGPT](https://chatgpt.com/?q=You're%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20(github.com%2Fnalediym%2Fmotion-from-scratch)%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Ask%20me%20questions%20first.%20If%20I'm%20still%20stuck%2C%20give%20one%20hint%20at%20a%20time%3A%20point%20at%20the%20test's%20expected%20vs%20actual%20values%2C%20then%20ask%20a%20concept%20question%2C%20then%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%20using%20different%20numbers%2C%20then%20point%20at%20the%20line%20to%20look%20at.%20Never%20go%20further%20than%20that.%20Explain%20what's%20happening%20under%20the%20hood%20(bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it).%20I'm%20on%20layer%2000%20(Pixels)%3A%20a%20framebuffer%3A%20memory%20you%20can%20see.%20Start%20by%20asking%20what%20I'm%20working%20on.%20I'll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
%%writefile src/00-pixels.js
// Layer 00 · Pixels — a framebuffer: memory you can see.
// Read docs/00-pixels.md first. Run `node tests/run.js 00` to check your work.
//
// A frame is a plain object:
//   { width, height, data }
// where `data` is a Uint8ClampedArray holding 4 numbers per pixel, row by row:
//
//   data:  [ R G B A | R G B A | R G B A | ... ]
//            pixel 0   pixel 1   pixel 2
//
// A color is an array [r, g, b] or [r, g, b, a], each 0–255.
// If alpha is missing, treat it as 255 (fully opaque).
(function (MS) {
  /**
   * Make a new frame, every pixel transparent black [0, 0, 0, 0].
   * Throw an Error if width or height is not a positive whole number.
   * @returns {{ width: number, height: number, data: Uint8ClampedArray }}
   */
  function createFrame(width, height) {
    throw MS.todo("createFrame");
  }

  /**
   * Where does pixel (x, y) start inside frame.data?
   * Return the index of its R value. (G, B and A follow right after it.)
   * Assume x and y are whole numbers inside the frame.
   */
  function pixelIndex(frame, x, y) {
    throw MS.todo("pixelIndex");
  }

  /**
   * Write `color` into pixel (x, y), replacing what was there.
   * - Round fractional coordinates down (2.7 → 2).
   * - If (x, y) is outside the frame, do nothing. Don't throw, and don't
   *   let x = width "wrap around" onto the next row.
   */
  function setPixel(frame, x, y, color) {
    throw MS.todo("setPixel");
  }

  /**
   * Read pixel (x, y) as a new array [r, g, b, a].
   * Round coordinates down like setPixel. Return null if outside the frame.
   */
  function getPixel(frame, x, y) {
    throw MS.todo("getPixel");
  }

  /**
   * Fill every pixel of the frame with `color`.
   */
  function clearFrame(frame, color) {
    throw MS.todo("clearFrame");
  }

  Object.assign(MS, { createFrame, pixelIndex, setPixel, getPixel, clearFrame });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
# Tests for layer 00: run after saving your code above
!node tests/run.js 00

In [ ]:
# Pictures drawn by your code
pictures("00")

## Layer 01 · Shapes & blending

_Deciding which pixels a shape covers, and what happens where shapes overlap._

**You'll write:** `hex`, `blendPixel`, `fillRect`, `fillCircle`, `mixColor`, `fillVerticalGradient`, `fillRadialGradient` in `src/01-shapes.js`
**Check it:** `node tests/run.js 01`
**Builds on:** layer 00

KineticIntro has a glowing radial background, words that fade in and out, and a light sweep that brightens the logo as it passes. All three are this layer: filling shapes, mixing colors, and **blending** a see-through color over what's already there.

### Under the hood

#### Colors as text: hex

Designers write colors like `#5B8CFF`. That's three bytes written in base 16 (hexadecimal), two digits each:

```text
   #  5B   8C   FF
      │    │    └─ blue  = 0xFF = 15×16 + 15 = 255
      │    └────── green = 0x8C =  8×16 + 12 = 140
      └─────────── red   = 0x5B =  5×16 + 11 =  91
```

JavaScript can read base-16 text for you (look up `parseInt`'s second argument). There are two common approaches: cut the string into three 2-character pieces, or read all six digits as one number and pull bytes out of it with bit shifts (`>>`) and masks (`& 255`). Either is fine. The second shows you how colors are often packed in memory.

#### Rasterizing: which pixels does a shape cover?

A rectangle from x = 2 to x = 6 is an idea with perfect edges. The screen only has whole pixels. Turning shapes into pixels is called **rasterization**, and the rule almost every renderer uses is: _a pixel belongs to the shape if its **center** is inside._

```text
        0     1     2     3     4     5     6     7
     ┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
     │  ·  │  ·  │  ■  │  ■  │  ■  │  ■  │  ·  │  ·  │   rect x = 2 .. 6
     └─────┴─────┴─────┴─────┴─────┴─────┴─────┴─────┘
                 ↑ centers at 2.5, 3.5, 4.5, 5.5 are inside
                                         ↑ 6.5 is not
```

For rectangles, testing centers is the same as rounding the edges, which is what the spec asks for.

Notice that the right edge is **exclusive**: a rectangle from 2 to 6 covers 2, 3, 4, 5, and not 6. This "half-open" rule means two rectangles that share an edge (one from 2 to 6, the next from 6 to 9) meet perfectly, with no gap and no pixel painted twice. You'll see why that matters in **Break it**.

#### Circles: distance from the center

A pixel is inside a circle if its center is no farther than `r` from the circle's center. That's Pythagoras: `dx² + dy² ≤ r²`. (You can skip the square root by comparing squared distances. Why does that give the same answer?)

```text
          · · · · · · · · ·
          · · · ■ ■ ■ · · ·         every ■ passes  dx² + dy² ≤ r²
          · · ■ ■ ■ ■ ■ · ·
          · · ■ ■ ● ■ ■ · ·         ● = center
          · · ■ ■ ■ ■ ■ · ·
          · · · ■ ■ ■ · · ·         ┌─────────┐  only test pixels
          · · · · · · · · ·         └─────────┘  inside this box
```

You *could* test every pixel in the frame. For a small circle on a 1080 × 1920 frame that's 2 million tests to draw maybe 300 pixels. Instead, only visit the circle's **bounding box**: from `cx − r` to `cx + r` in each direction.

Jagged edges like these are called **aliasing**. Real renderers soften them with **antialiasing**: a pixel that's 40% covered gets 40% of the color. See **Going further**.

#### Alpha blending: painting with see-through paint

Here's the big one. When you paint a half-transparent red over a blue pixel, the result is a mix. When you paint it over an empty, fully transparent pixel, it should still be red, just see-through.

This was worked out in 1984 by Thomas Porter and Tom Duff at Lucasfilm, in a paper called "Compositing Digital Images". It's how film effects layered spaceships over backgrounds, and how every browser layers one element over another. Their "over" operation, with alphas written as 0 to 1:

```text
  a = alpha of the new color (source)     b = alpha of the existing pixel (destination)

  outAlpha = a + b × (1 − a)
  outRGB   = ( newRGB × a  +  oldRGB × b × (1 − a) ) / outAlpha
```

Read it as coverage. The new paint covers fraction `a` of the pixel. Of what's left, `(1 − a)`, the old pixel covers fraction `b`. The color is the average of the two, weighted by how much each one covers. The **divide by outAlpha** turns "amount of light" back into "color".

```text
  50% red over opaque blue:              50% red over nothing:
  outAlpha = 0.5 + 1 × 0.5 = 1           outAlpha = 0.5 + 0 = 0.5
  red  = (255×0.5 + 0) / 1   ≈ 128       red  = (255×0.5) / 0.5 = 255   ← still pure red
  blue = (0 + 255×1×0.5) / 1 ≈ 127       alpha = 0.5 → 128
```

Colors stored this way are called **straight alpha**. Many renderers store colors already multiplied by alpha instead (**premultiplied alpha**), because then the formula needs no division. That choice has consequences, which you'll explore below.

#### Gradients: one color per position

A gradient answers one question per pixel: _how far along am I, from 0 to 1?_ Then it mixes the two colors by that amount:

```text
  mixColor(a, b, t) = a + (b − a) × t      (for each of R, G, B and A)

  t = 0          t = 0.25        t = 0.5         t = 0.75        t = 1
  ██ a           ██              ██ halfway      ██              ██ b
```

A vertical gradient gets `t` from the row. A radial gradient gets `t` from distance to a center point. Dividing each axis by its own radius (`rx`, `ry`) stretches the circle into an ellipse.

That's exactly what CSS does. KineticIntro's background is written in CSS as:

```text
  radial-gradient(70% 60% at 50% 38%, #1A2740, #0B0D12)
                  └rx  ry┘   └cx  cy┘   inner     outer
  on a 1080 × 1920 frame:  rx = 756   ry = 1152   cx = 540   cy = 729.6
```

When your tests pass, `tests.html` draws that exact background. It's the first real piece of the recreation.

### Your turn

1. **hex.** What should happen with `"#FFF"`, `"blue"`, or `"#12345G"`? A regular expression like `/^#?[0-9a-f]{6}$/i` is one way to check. Can you explain each part of it?
2. **blendPixel.** Turn both alphas into 0–1 first. Handle the case where `outAlpha` is 0. What would dividing by 0 put in the pixel?
3. **fillRect.** Round the four edges, then clip them to the frame *before* looping. Why is clipping first better than letting `blendPixel` ignore off-screen pixels?
4. **fillCircle.** Compute the bounding box, clip it, then test each pixel's center.
5. **mixColor, fillVerticalGradient, fillRadialGradient.** Where is the middle of row `y`? Why sample there instead of at its top edge?

### Predict, then run

1. How many pixels does `fillRect(f, 0.4, 0, 2, 1, c)` cover? And `fillRect(f, 0.6, 0, 2, 1, c)`?
2. A circle of radius 5 centered on a pixel center. The area of a circle is πr² ≈ 78.5. How many pixels will be filled?
3. `[255, 0, 0, 128]` blended onto `[0, 0, 0, 0]`. What are the four numbers afterwards?

<details><summary>Answers</summary>

1. Two each. Edges 0 → 2, and 1 → 3.
2. 81. Close to 78.5 but not equal: pixels are all-or-nothing, so the count wobbles around the true area.
3. `[255, 0, 0, 128]`. The color stays pure red; only the alpha says "half there".

</details>

### Break it

1. **Forget the division.** Remove `/ outAlpha` from `blendPixel`. Which tests fail? Blend 50% red onto a transparent pixel and look at the result: it's dark red, `[128, 0, 0, 128]`. When that pixel is later layered over something bright, it shows up as a dark fringe. Dark halos around cut-out images are this exact bug, in real software.
2. **Make the right edge inclusive.** Change `fillRect` so it covers `right` too. Then draw two half-transparent rectangles side by side, one from 0 to 10 and the next from 10 to 20. Look closely at x = 10. What happened, and why does it look like a seam?
3. **Skip the bounding box.** Make `fillCircle` loop over the whole frame. Time drawing 1,000 small circles on a 1080 × 1920 frame, before and after, with `console.time`.

### Checkpoint

- `node tests/run.js 01` shows 17 of 17 passing.
- `tests.html` shows overlapping blended shapes, and a tall dark-blue glowing background. That background is KineticIntro's.
- You can explain why half-transparent red over nothing stays red.

### In the real world

- Canvas's `globalAlpha` and CSS `opacity` both use Porter–Duff "over".
- Browsers usually keep pixels premultiplied internally. That's why a canvas pixel with very low alpha can come back from `getImageData` with slightly different colors than you put in: precision was lost in the multiply and divide.
- Motion Studio's `KineticIntro.tsx` paints its background with the CSS gradient above, then adds a second radial glow and a dark vignette on top. Those are three layers blended "over" each other, which you can now do by hand.

### Study buddy prompts

- "I'm on layer 01 of Motion from Scratch. Without giving me code, walk me through Porter–Duff 'over' by asking me what should happen in three cases I can check by hand."
- "Ask me to predict which pixels fillRect(1.5, 0, 2, 1) covers, then tell me whether my reasoning (not just my answer) was right."
- "Explain straight versus premultiplied alpha with an analogy, then give me a break-it experiment that shows the difference."

### Going further (optional)

- **Antialiased circles.** Instead of in-or-out, use coverage `clamp(r − distance + 0.5, 0, 1)` and multiply it into the color's alpha before blending. Compare the edges at scale in `tests.html`.
- **Premultiplied mode.** Write `blendPremultiplied` where colors are stored already multiplied by alpha. Show that the formula no longer needs a division, and find which test values it can't reproduce exactly after 8-bit rounding.


### ✏️ Your code for layer 01

Edit the cell below, then run it. The first line saves it as `src/01-shapes.js`. Then run the tests and pictures under it.

**Stuck?** Ask a study buddy that gives hints, not answers: [open Claude](https://claude.ai/new?q=You're%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20(github.com%2Fnalediym%2Fmotion-from-scratch)%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Ask%20me%20questions%20first.%20If%20I'm%20still%20stuck%2C%20give%20one%20hint%20at%20a%20time%3A%20point%20at%20the%20test's%20expected%20vs%20actual%20values%2C%20then%20ask%20a%20concept%20question%2C%20then%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%20using%20different%20numbers%2C%20then%20point%20at%20the%20line%20to%20look%20at.%20Never%20go%20further%20than%20that.%20Explain%20what's%20happening%20under%20the%20hood%20(bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it).%20I'm%20on%20layer%2001%20(Shapes%20%26%20blending)%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Start%20by%20asking%20what%20I'm%20working%20on.%20I'll%20paste%20my%20code%20and%20test%20output%20next.) · [open ChatGPT](https://chatgpt.com/?q=You're%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20(github.com%2Fnalediym%2Fmotion-from-scratch)%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Ask%20me%20questions%20first.%20If%20I'm%20still%20stuck%2C%20give%20one%20hint%20at%20a%20time%3A%20point%20at%20the%20test's%20expected%20vs%20actual%20values%2C%20then%20ask%20a%20concept%20question%2C%20then%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%20using%20different%20numbers%2C%20then%20point%20at%20the%20line%20to%20look%20at.%20Never%20go%20further%20than%20that.%20Explain%20what's%20happening%20under%20the%20hood%20(bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it).%20I'm%20on%20layer%2001%20(Shapes%20%26%20blending)%3A%20rectangles%2C%20circles%2C%20alpha%2C%20gradients.%20Start%20by%20asking%20what%20I'm%20working%20on.%20I'll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
%%writefile src/01-shapes.js
// Layer 01 · Shapes & blending — rectangles, circles, alpha, gradients.
// Read docs/01-shapes.md first. Run `node tests/run.js 01` to check your work.
// You can (and should) use your layer 00 functions here.
(function (MS) {
  /**
   * Turn a CSS-style hex string into a color array.
   *   hex("#5B8CFF")      → [91, 140, 255, 255]
   *   hex("5b8cff", 128)  → [91, 140, 255, 128]
   * Accept upper or lower case, with or without "#".
   * Throw an Error for anything that isn't exactly 6 hex digits.
   */
  function hex(str, alpha = 255) {
    throw MS.todo("hex");
  }

  /**
   * Paint `color` OVER pixel (x, y), respecting its alpha ("source-over").
   * With a = alpha of the new color, b = alpha of the existing pixel,
   * both as 0–1:
   *
   *   outAlpha = a + b × (1 − a)
   *   outRGB   = (newRGB × a + oldRGB × b × (1 − a)) / outAlpha
   *
   * Write outAlpha back as 0–255. If outAlpha is 0, the pixel becomes
   * [0, 0, 0, 0]. Round coordinates down; ignore pixels outside the frame.
   */
  function blendPixel(frame, x, y, color) {
    throw MS.todo("blendPixel");
  }

  /**
   * Blend a w × h rectangle whose top-left corner is (x, y).
   * Round the edges to whole pixels first:
   *   left = round(x), right = round(x + w)   (right is exclusive)
   *   top  = round(y), bottom = round(y + h)  (bottom is exclusive)
   * Clip to the frame, so a rectangle hanging off the edge is fine.
   */
  function fillRect(frame, x, y, w, h, color) {
    throw MS.todo("fillRect");
  }

  /**
   * Blend a circle with center (cx, cy) and radius r.
   * A pixel is inside when its CENTER, (px + 0.5, py + 0.5), is at
   * distance <= r from (cx, cy). Only visit pixels in the circle's
   * bounding box, not the whole frame.
   */
  function fillCircle(frame, cx, cy, r, color) {
    throw MS.todo("fillCircle");
  }

  /**
   * Mix two colors: t = 0 gives a, t = 1 gives b, 0.5 is halfway.
   * Mix all four channels (missing alpha = 255). Don't round.
   */
  function mixColor(a, b, t) {
    throw MS.todo("mixColor");
  }

  /**
   * Fill the whole frame from `top` color to `bottom` color.
   * Row y uses t = (y + 0.5) / height, so you sample each row at its middle.
   * Overwrite pixels (setPixel), don't blend.
   */
  function fillVerticalGradient(frame, top, bottom) {
    throw MS.todo("fillVerticalGradient");
  }

  /**
   * Fill the whole frame with an elliptical radial gradient, like CSS
   * `radial-gradient(rx ry at cx cy, inner, outer)`.
   * For each pixel center (x + 0.5, y + 0.5):
   *   d = sqrt( ((x + 0.5 − cx) / rx)² + ((y + 0.5 − cy) / ry)² )
   *   t = min(d, 1)
   * and write mixColor(inner, outer, t). Overwrite, don't blend.
   */
  function fillRadialGradient(frame, cx, cy, rx, ry, inner, outer) {
    throw MS.todo("fillRadialGradient");
  }

  Object.assign(MS, { hex, blendPixel, fillRect, fillCircle, mixColor, fillVerticalGradient, fillRadialGradient });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
# Tests for layer 01: run after saving your code above
!node tests/run.js 01

In [ ]:
# Pictures drawn by your code
pictures("01")

## Layer 02 · Glyphs

_A font is a lookup table from characters to tiny pictures._

**You'll write:** `drawGlyph`, `measureText`, `drawText` in `src/02-glyphs.js`
**Check it:** `node tests/run.js 02`
**Builds on:** layers 00–01 (especially `fillRect`)

KineticIntro is mostly words: **DESIGN.**, **ANIMATE.**, **RENDER.**, then **MOTION·STUDIO**. In this layer you'll draw text using nothing but the rectangles you already have and a font made of `#` and `.` characters. In layer 07 you'll switch to the browser's real fonts, and this layer is how you'll understand what the browser is doing for you.

### Under the hood

#### Two kinds of fonts

**Bitmap fonts** store each character as a grid of on/off pixels. Arcade machines, early terminals, and the first Macintosh used them; Susan Kare drew the original Mac fonts on graph paper in 1983–84. They're simple and fast, but they only look right at one size.

**Outline fonts** (TrueType, OpenType) store each character as curves, so they can be drawn at any size. Drawing one means rasterizing the curves into pixels with antialiasing, which is the "which pixels does this shape cover?" problem from layer 01, done very well. That's what the browser does when you write `font-size: 150px`.

You're building a bitmap font renderer because it shows the whole pipeline with no magic: _character → lookup → grid → pixels_.

#### The font file

`lib/font5x7.js` is the font. Every glyph is 7 rows of 5 cells:

```text
   MS.FONT_5x7.N  =               drawn at scale 3:
   [ "#...#",                     ■■■         ■■■
     "##..#",                     ■■■■■■      ■■■
     "#.#.#",                     ■■■   ■■■   ■■■
     "#..##",          ──►        ■■■      ■■■■■■
     "#...#",                     ■■■         ■■■
     "#...#",                     ■■■         ■■■
     "#...#" ]                    ■■■         ■■■
                                  (each # becomes a 3 × 3 block)
```

Reading it is a nested loop: for each row `r`, for each column `c`, if the cell is `#`, fill a `scale × scale` square at `(x + c × scale, y + r × scale)`. Scaling by repeating each pixel as a block like this is called **nearest-neighbor** scaling. It's why pixel art stays crisp.

#### Laying out a line

Text layout is the job of deciding where each glyph goes. Ours is the simplest possible version:

```text
   x                                                    one glyph = 5 cells
   │◄─ 5 ─►│◄1►│◄─ 5 ─►│◄1►│◄─ 5 ─►│                    tracking  = gap between glyphs
   ┌───────┐   ┌───────┐   ┌───────┐
   │   H   │   │   I   │   │   !   │                    width = 3 × 5 + 2 × 1 = 17 cells
   └───────┘   └───────┘   └───────┘                          (× scale for pixels)
   ◄──────────── measureText = 17 ───────────►
```

- **Advance** is how far you move right after each glyph: its width plus the tracking.
- **Tracking** is uniform extra space between all letters. (Kerning, which tightens specific pairs like "AV", is a different thing. Our font skips it.)
- Notice that tracking only goes *between* glyphs. The last glyph doesn't get a gap after it.

**Alignment needs measuring first.** To center text on `x`, you need its total width before drawing the first glyph, so the left edge is `x − width / 2`. Real layout engines work in the same two passes: measure everything, then place everything. KineticIntro centers every word, so you'll use this constantly.

#### Characters are trickier than they look

- **Case.** The font has only uppercase. Mapping `"g"` to `"G"` is a tiny case of what real fonts do with **fallback**: when the font lacks a character, find a substitute.
- **Missing characters.** When there's no substitute, show **tofu**: an empty box. (Google's font family Noto is named for "no more tofu".) Tofu is ugly on purpose. It tells you something is missing instead of silently skipping it.
- **What counts as one character?** `"MOTION·STUDIO"` has 13 characters, and `·` is one of them. But try `"👋".length` in the console: it says 2. JavaScript strings count 16-bit units, and many emoji take two. `for (const ch of text)` and `[...text]` walk real characters. A plain `for (let i = 0; i < text.length; i++)` splits that emoji in half. Pick your loop carefully.

#### Sizing for the real video

KineticIntro sets its words at a font size of 150 px in a 1080 px wide frame. Font size is the height of the font's whole design box, not of a capital letter. In Inter, the font it uses, capitals are about 73% of that, so roughly 110 px tall. Our capitals are 7 cells tall, so a scale of about 16 matches. Can **ANIMATE.** fit across 1080 pixels at scale 16 with tracking 1? How large can the scale get before it doesn't? Work it out with `measureText` once it passes.

### Your turn

1. **measureText.** Start here: it's pure arithmetic and has the clearest tests. What should it return for the empty string? For one glyph?
2. **drawGlyph.** Look up the glyph (uppercase first). If it's missing, draw tofu, the outer ring of the 5 × 7 grid. Otherwise loop rows and columns and fill the `#` cells with `fillRect`.
3. **drawText.** Measure once, work out the starting x from `align`, then walk the characters, advancing by one glyph plus tracking each time. Spaces advance without drawing.

### Predict, then run

1. How wide is `"DESIGN."` at scale 4 with tracking 1?
2. What does `drawGlyph(f, "Ä", 0, 0, 1, white)` draw?
3. `drawText(f, "II", 20, 0, 1, white, { align: "center" })`: the width is 11, so the left edge is 14.5. Which pixel column gets the first ink, and why that one?

<details><summary>Answers</summary>

1. 164 pixels: 7 glyphs × 5 cells × 4, plus 6 gaps × 1 × 4.
2. Tofu. "Ä" isn't in the font, and uppercasing it doesn't help.
3. Column 15. Your `fillRect` rounds edges, and `round(14.5)` is 15. Text lands on whole pixels, just like rectangles.

</details>

### Break it

1. **Put tracking after every glyph**, including the last, in `measureText`. Which tests fail? Center a word and look: it drifts right by half a gap. Real UI text that looks "slightly off-center" is often this bug.
2. **Loop by index.** Change `drawText` to `for (let i = 0; i < text.length; i++)` and draw `"HI 👋"`. How many tofu boxes appear for the one emoji, and why?
3. **Skip the uppercase step.** Draw `"Design."`. What appears?

### Checkpoint

- `node tests/run.js 02` shows 11 of 11 passing.
- `tests.html` shows **DESIGN.** in big pixel letters, plus an alignment test with a pink center line and one tofu box.
- You can explain why centering text needs `measureText` before the first glyph is drawn.

### In the real world

- Canvas's [`measureText`](https://developer.mozilla.org/en-US/docs/Web/API/CanvasRenderingContext2D/measureText) and `textAlign` do what you just did, for outline fonts, with kerning and shaping on top.
- Text shaping engines like HarfBuzz handle what our font ignores: kerning, ligatures, scripts that join letters, and right-to-left text.
- In Motion Studio's `KineticIntro.tsx`, each word is centered with flexbox and drawn by the browser at a 150 px font size. In layer 07 you'll switch to real fonts too, so you can compare your pixel version with the browser's directly.

### Study buddy prompts

- "I'm on layer 02 of Motion from Scratch. Ask me questions until I can explain the difference between advance, tracking and kerning. No code."
- "Here's my measureText. Don't fix it; give me one input where it's wrong and let me work out why."
- "Explain how outline fonts get rasterized, connecting it to the pixel-center rule I learned in layer 01."

### Going further (optional)

- **Design your own glyphs.** Add lowercase letters or symbols to a copy of the font. What's the smallest grid where lowercase "g" still has a readable descender?
- **Outlines.** Draw text with a 1-cell dark outline by drawing it shifted in 8 directions, then on top. Count how many `fillRect` calls that costs, and think about why real renderers don't do it this way.


### ✏️ Your code for layer 02

Edit the cell below, then run it. The first line saves it as `src/02-glyphs.js`. Then run the tests and pictures under it.

**Stuck?** Ask a study buddy that gives hints, not answers: [open Claude](https://claude.ai/new?q=You're%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20(github.com%2Fnalediym%2Fmotion-from-scratch)%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Ask%20me%20questions%20first.%20If%20I'm%20still%20stuck%2C%20give%20one%20hint%20at%20a%20time%3A%20point%20at%20the%20test's%20expected%20vs%20actual%20values%2C%20then%20ask%20a%20concept%20question%2C%20then%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%20using%20different%20numbers%2C%20then%20point%20at%20the%20line%20to%20look%20at.%20Never%20go%20further%20than%20that.%20Explain%20what's%20happening%20under%20the%20hood%20(bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it).%20I'm%20on%20layer%2002%20(Glyphs)%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Start%20by%20asking%20what%20I'm%20working%20on.%20I'll%20paste%20my%20code%20and%20test%20output%20next.) · [open ChatGPT](https://chatgpt.com/?q=You're%20my%20study%20buddy%20for%20Motion%20from%20Scratch%20(github.com%2Fnalediym%2Fmotion-from-scratch)%2C%20a%20course%20where%20I%20build%20a%20video%20engine%20from%20single%20pixels%20up%20in%20JavaScript.%20Rules%3A%20never%20write%2C%20finish%20or%20fix%20my%20code%2C%20even%20if%20I%20ask.%20Ask%20me%20questions%20first.%20If%20I'm%20still%20stuck%2C%20give%20one%20hint%20at%20a%20time%3A%20point%20at%20the%20test's%20expected%20vs%20actual%20values%2C%20then%20ask%20a%20concept%20question%2C%20then%20explain%20with%20an%20analogy%20or%20ASCII%20diagram%20using%20different%20numbers%2C%20then%20point%20at%20the%20line%20to%20look%20at.%20Never%20go%20further%20than%20that.%20Explain%20what's%20happening%20under%20the%20hood%20(bytes%2C%20rounding%2C%20math%2C%20how%20browsers%20do%20it).%20I'm%20on%20layer%2002%20(Glyphs)%3A%20text%20from%20a%20hand-made%20bitmap%20font.%20Start%20by%20asking%20what%20I'm%20working%20on.%20I'll%20paste%20my%20code%20and%20test%20output%20next.)

In [ ]:
%%writefile src/02-glyphs.js
// Layer 02 · Glyphs — text from a hand-made bitmap font.
// Read docs/02-glyphs.md first. Run `node tests/run.js 02` to check your work.
//
// The font lives in lib/font5x7.js as MS.FONT_5x7: an object mapping a
// character to 7 strings of 5 characters, "#" for ink and "." for empty.
// Use fillRect from layer 01 to draw each "#" as a scale × scale block.
(function (MS) {
  const GLYPH_W = 5;
  const GLYPH_H = 7;

  /**
   * Draw one character with its top-left corner at (x, y).
   * Each "#" cell becomes a filled square of size `scale` × `scale`.
   * - The font only has uppercase letters: "a" must draw like "A".
   * - A character missing from the font draws "tofu": a hollow 5 × 7 box
   *   (the outer ring of cells filled, the inside empty).
   */
  function drawGlyph(frame, ch, x, y, scale, color) {
    throw MS.todo("drawGlyph");
  }

  /**
   * How wide is `text` in pixels?
   * Each glyph is GLYPH_W cells wide, and `tracking` empty cells sit
   * BETWEEN glyphs (not after the last one). Multiply cells by `scale`.
   *   measureText("AB", 1)    → 5 + 1 + 5 = 11
   *   measureText("AB", 3)    → 33
   *   measureText("", 3)      → 0
   */
  function measureText(text, scale, tracking = 1) {
    throw MS.todo("measureText");
  }

  /**
   * Draw a line of text and return its width (same number measureText gives).
   * opts.tracking : empty cells between glyphs (default 1)
   * opts.align    : "left" (default) — x is the left edge
   *                 "center"         — x is the middle
   *                 "right"          — x is the right edge
   * A space draws nothing but still takes up a glyph's width.
   */
  function drawText(frame, text, x, y, scale, color, opts = {}) {
    throw MS.todo("drawText");
  }

  Object.assign(MS, { GLYPH_W, GLYPH_H, drawGlyph, measureText, drawText });
})(globalThis.MS = globalThis.MS || {});


In [ ]:
# Tests for layer 02: run after saving your code above
!node tests/run.js 02

In [ ]:
# Pictures drawn by your code
pictures("02")

## What's next

Layers 03–10 are on the way: time, interpolation, easing, springs, the hand-off to canvas, scenes, the KineticIntro rebuild, and MP4 export. Watch [the repo](https://github.com/nalediym/motion-from-scratch) for updates.